In [1]:
import pandas as pd

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
data = pd.read_csv("../data/train-2025.csv")
data.head()

/var/folders/3c/kxfpvzq56558my2fcggnzrn40000gn/T/ipykernel_34036/3195205904.py:1: DtypeWarning: Columns (0: def_date) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("../data/train-2025.csv")


,Unnamed: 0,id,stmt_date,HQ_city,legal_struct,ateco_sector,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,...,inc_extraord,taxes,profit,days_rec,ebitda,roa,roe,wc_net,margin_fin,cf_operations
0,17,520288,2011-12-31,28.0,SRL,14.0,NaN,2011,67537.0,1137566.0,...,-309.0,-80959.0,-81288.0,NaN,6318.0,-3.81,-28.03,496258.0,-917029.0,-849.0
1,18,520288,2008-12-31,28.0,SRL,14.0,NaN,2008,256438.0,1181416.0,...,-678.0,-94622.0,-107382.0,NaN,46088.0,-2.76,NaN,97952.0,NaN,-3881.0
2,19,520288,2009-12-31,28.0,SRL,14.0,NaN,2009,194046.0,1152014.0,...,4224.0,-74235.0,-77819.0,NaN,67611.0,-2.17,NaN,-210671.0,NaN,32618.0
3,21,520288,2012-12-31,28.0,SRL,14.0,NaN,2012,15195.0,1116938.0,...,3634.0,-250786.0,-250786.0,NaN,-161478.0,-12.99,NaN,367892.0,-1094962.0,-168907.0
4,22,520288,2007-12-31,28.0,SRL,14.0,NaN,2007,126603.0,1127807.0,...,820.0,92192.0,36733.0,NaN,153060.0,6.20,52.43,-317007.0,-1184970.0,80039.0


In [3]:
def make_label(df, publish_month=5, horizon_months=12, cutoff=None):
    """12-month default label. The FY-Y report is assumed to be published before the end of month `publish_month` of year Y+1; label = 1 if def_date happens in
    (publish date, publish date + horizon_months], else 0.

    Input : df (pd.DataFrame) with fs_year (int) and def_date.
            cutoff: last observable default date, defaults to df["def_date"].max().
    Output: pd.DataFrame = df columns + "label" (0/1), with unlabelable rows dropped.
    """
    pub = pd.to_datetime((df["fs_year"] + 1).astype(str) + f"-{publish_month:02d}-01") + pd.offsets.MonthEnd(0)
    end = pub + pd.DateOffset(months=horizon_months)
    dfl = pd.to_datetime(df["def_date"])
    cutoff = dfl.max() if cutoff is None else pd.Timestamp(cutoff)

    keep = (dfl.isna() | (dfl > pub)) & (end <= cutoff)
    return df[keep].assign(label=((dfl > pub) & (dfl <= end))[keep].astype(int))

### Why some rows cannot be labeled (188,536 of 1,023,552 rows are dropped)

Assumption: the FY Y report is published at the end of May of Y+1 (`publish_month=5`), so FY2011's window is 2012-06-01 to 2013-05-31, which the data fully covers. 188,536 rows have no valid label:

| Reason | Rows |
|---|---|
| **Window runs past the data.** The last observed default is 2013-05-31. FY2012's window (2013-06 to 2014-05) is entirely outside the data. | 185,639 (all FY2012) |
| **Defaulted before the report was published** (FY2007-2011 rows). The firm defaulted between fiscal year-end and the publish date (May 31 of Y+1), so the bank could never have used this statement. | 2,897 |
| **Total** | **188,536** |

Notes:
- For FY2012, `def_date = NaN` is ambiguous: the firm may truly not have defaulted, or may have defaulted after the data ends. Labeling these 0 would understate the default rate, so they are dropped, not set to 0.
- 3,547 rows in total defaulted before their publish date (650 of them are FY2012 rows, already counted in the first group). The default is not lost: for 2,949 of them the FY Y-1 row of the same firm exists and carries `label = 1`. The other 598 have no earlier statement.
- Remaining labeled data: FY2007-2011, 835,016 rows (823,680 zeros, 11,336 ones).

In [4]:
data = make_label(data)
print(data.shape)
print(data.groupby("fs_year")["label"].agg(["count", "mean"]))  # count = rows, mean = default rate

(835016, 45)
          count      mean
fs_year                  
2007     144251  0.008735
2008     161383  0.012052
2009     170549  0.015755
2010     174674  0.016494
2011     184159  0.013917


In [5]:
sub = data[["def_date", "fs_year", "label"]]
sub[sub["label"] == 1]

,def_date,fs_year,label
957960,2008-10-31,2007,1
957961,2008-07-28,2007,1
957962,2008-12-22,2007,1
957963,2008-11-19,2007,1
957965,2008-06-05,2007,1
...,...,...,...
1013334,2013-04-29,2011,1
1013344,2013-01-30,2011,1
1013350,2013-04-22,2011,1
1013359,2013-02-05,2011,1


In [6]:
data["label"].value_counts()

label
0    823680
1     11336
Name: count, dtype: int64